# P5 — Regime + Time-of-day + P&L control + ML ensemble

**Slug**: `crypto_intraday/40_regime_tod_pnl_ml`

**Status**: in_progress

**Goal**: Address the four open questions left by the v1 PM report:
1. **Cross-regime validation** — re-run all strategies on 2022-Q1 (bear), 2023-H1 (chop), 2024-H1 (bull) and report per regime.
2. **Asia vs US time-of-day exploitation** — decompose performance into Asia (0-8 UTC), Europe (8-13 UTC), US (13-21 UTC), Late (21-24 UTC) and check whether different sessions favor different signals.
3. **P&L control on long-holding strategies** (`funding_momentum` had 89-day mean holding) — add stop-loss + profit-take + time-out; quantify whether risk control improves or hurts per-regime Sharpe.
4. **ML signal ensemble** — use strategy signals (not raw features) as input to a meta-classifier; promote only if it beats the best single strategy in every regime.

All three regime windows fall inside the train slice (≤ 2024-07-01) — PM holdout remains untouched.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
from dataclasses import dataclass
from itertools import product

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb

from alpha_lab.backtest.metrics import summary
from alpha_lab.backtest.vector import run_backtest
from alpha_lab.data.holdout import PMHoldout, access_summary_for_report
from alpha_lab.data.loaders import binance_vision as bv
from alpha_lab.features import intraday as f
from alpha_lab.features.transforms import Standardizer
from alpha_lab.ml.cv import WalkForwardSplit
from alpha_lab.utils.config import load_config

pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.width', 220)

In [ ]:
cfg = load_config('crypto_intraday')
holdout = PMHoldout.from_config('crypto_intraday')
SYMBOLS = ['BTCUSDT', 'ETHUSDT']
CAP = 0.5
BPY_1H = 365 * 24

REGIMES = {
    'bear_2022Q1':  ('2022-01-01', '2022-04-01'),
    'chop_2023H1':  ('2023-01-01', '2023-07-01'),
    'bull_2024H1':  ('2024-01-01', '2024-07-01'),
}
for name, (s, e) in REGIMES.items():
    print(f'{name}: [{s}, {e})')

## 1. Load + resample (1m → 1h) for each regime

First call pulls fresh ZIPs from data.binance.vision (~2-5 min); subsequent calls hit the parquet cache.

In [ ]:
def _agg(df: pd.DataFrame, off: str) -> pd.DataFrame:
    return df.resample(off, label='right', closed='right').agg({
        'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last',
        'volume': 'sum', 'quote_volume': 'sum', 'n_trades': 'sum',
        'taker_buy_base': 'sum',
    }).dropna(how='all')

regime_data: dict[str, dict] = {}
for name, (s, e) in REGIMES.items():
    panel = bv.load_klines(SYMBOLS, '1m', start=s, end=e, market='perp', holdout=holdout)
    funding = bv.load_funding(SYMBOLS, start=s, end=e, holdout=holdout)
    ohlcv_1h = {sym: _agg(df, '1h') for sym, df in panel.frames.items()}
    prices_1h = pd.DataFrame({sym: o['close'] for sym, o in ohlcv_1h.items()}).dropna()
    common = prices_1h.index
    ohlcv_1h = {sym: o.loc[common] for sym, o in ohlcv_1h.items()}
    regime_data[name] = {
        'ohlcv_1h': ohlcv_1h,
        'prices': prices_1h,
        'funding': funding,
        'common': common,
    }
    print(f'{name}: {len(common):,} 1h bars  funding={len(funding)}')
    if len(common):
        btc_ret = float(prices_1h['BTCUSDT'].iloc[-1] / prices_1h['BTCUSDT'].iloc[0] - 1)
        eth_ret = float(prices_1h['ETHUSDT'].iloc[-1] / prices_1h['ETHUSDT'].iloc[0] - 1)
        print(f'   BTC return: {btc_ret:+.2%}  ETH return: {eth_ret:+.2%}')

## 2. Strategy factory (same as P2, now parametrized over regime)

In [ ]:
def build_strategies(rd: dict) -> dict[str, callable]:
    ohlcv = rd['ohlcv_1h']
    prices = rd['prices']
    funding = rd['funding']
    common = rd['common']
    fz = f.funding_zscore(funding, 30).reindex(common, method='ffill')
    fc = f.funding_cumulative(funding, 10).reindex(common, method='ffill')

    def funding_contrarian():
        w = {}
        for sym in SYMBOLS:
            z = fz[sym].fillna(0.0)
            sig = pd.Series(0.0, index=z.index)
            sig[z > 1.0] = -1.0
            sig[z < -1.0] = 1.0
            w[sym] = sig * CAP
        return pd.DataFrame(w).fillna(0.0)

    def funding_momentum():
        w = {}
        for sym in SYMBOLS:
            cum = fc[sym].fillna(0.0)
            w[sym] = np.sign(cum) * CAP
        return pd.DataFrame(w).fillna(0.0)

    def bollinger_mr_threshold():
        w = {}
        for sym in SYMBOLS:
            bb = f.bollinger_pct_b(ohlcv[sym]['close'], 20)
            sig = pd.Series(0.0, index=bb.index)
            sig[bb < 0.1] = 1.0
            sig[bb > 0.9] = -1.0
            w[sym] = sig * CAP
        return pd.DataFrame(w).fillna(0.0)

    def rsi_mr_threshold():
        w = {}
        for sym in SYMBOLS:
            rsi = f.rsi(ohlcv[sym]['close'], 14)
            sig = pd.Series(0.0, index=rsi.index)
            sig[rsi < 30] = 1.0
            sig[rsi > 70] = -1.0
            w[sym] = sig * CAP
        return pd.DataFrame(w).fillna(0.0)

    def spread_z_pair_trade():
        z = f.spread_zscore(prices['BTCUSDT'], prices['ETHUSDT'], 48)
        sig = -z.clip(-2.0, 2.0) / 2.0
        return pd.DataFrame({'BTCUSDT': sig * CAP, 'ETHUSDT': -sig * CAP}).fillna(0.0)

    def beta_residual_mr():
        br = f.rolling_beta_residual(prices['ETHUSDT'], prices['BTCUSDT'], 48)
        z = br['residual'] / br['residual'].rolling(48, min_periods=12).std()
        eth_w = -z.clip(-2.0, 2.0) / 2.0 * CAP
        return pd.DataFrame({
            'BTCUSDT': pd.Series(0.0, index=common),
            'ETHUSDT': eth_w,
        }).fillna(0.0)

    return {
        'funding_contrarian':     funding_contrarian,
        'funding_momentum':       funding_momentum,
        'bollinger_mr_threshold': bollinger_mr_threshold,
        'rsi_mr_threshold':       rsi_mr_threshold,
        'spread_z_pair_trade':    spread_z_pair_trade,
        'beta_residual_mr':       beta_residual_mr,
    }

## 3. Per-regime full backtest (perp_stress costs)

In [ ]:
@dataclass(frozen=True)
class CostScenario:
    name: str
    fee_bps: float
    slip_bps: dict
    use_funding: bool

STRESS = CostScenario(
    name='perp_stress', fee_bps=5.0,
    slip_bps={'BTCUSDT': 3.0, 'ETHUSDT': 5.0}, use_funding=True,
)

def holding_period_hours(weights: pd.DataFrame, bar_minutes: int = 60) -> float:
    holds = []
    for sym in weights.columns:
        w = weights[sym]
        non_flat = int((w != 0).sum())
        changes = int((w.diff() != 0).sum())
        if changes == 0:
            holds.append(non_flat * bar_minutes / 60.0)
        else:
            holds.append((non_flat / changes) * bar_minutes / 60.0)
    return float(np.mean(holds))

regime_rows = []
regime_results: dict[tuple[str, str], object] = {}
for regime_name, rd in regime_data.items():
    strategies = build_strategies(rd)
    for strat_name, fn in strategies.items():
        w = fn().reindex(rd['common']).fillna(0.0)
        res = run_backtest(
            w, rd['prices'],
            costs_bps=STRESS.fee_bps, slippage_bps=STRESS.slip_bps,
            funding=rd['funding'] if STRESS.use_funding else None,
            bars_per_year=BPY_1H,
        )
        regime_results[(regime_name, strat_name)] = res
        stats = summary(res.returns, periods=BPY_1H)
        regime_rows.append({
            'regime': regime_name, 'strategy': strat_name,
            'net_total': float((1 + res.returns).prod() - 1),
            'Sharpe':    stats.get('Sharpe', float('nan')),
            'Sortino':   stats.get('Sortino', float('nan')),
            'Calmar':    stats.get('Calmar', float('nan')),
            'MaxDD':     stats.get('MaxDD', float('nan')),
            'avg_hold_h': holding_period_hours(w),
        })
rg_df = pd.DataFrame(regime_rows)
rg_df

In [ ]:
# Pivots — Sharpe and net by (strategy, regime)
print('=== Sharpe by (strategy, regime), perp_stress ===')
print(rg_df.pivot(index='strategy', columns='regime', values='Sharpe')
          [['bear_2022Q1','chop_2023H1','bull_2024H1']])
print()
print('=== Net total return by (strategy, regime), perp_stress ===')
print(rg_df.pivot(index='strategy', columns='regime', values='net_total')
          [['bear_2022Q1','chop_2023H1','bull_2024H1']])

## 4. Time-of-day decomposition

Bucket every bar's NET return into Asia (0-8 UTC), Europe (8-13 UTC), US (13-21 UTC), Late (21-24 UTC). Sharpe is computed per session × regime × strategy.

In [ ]:
def session_of(ts: pd.DatetimeIndex) -> pd.Series:
    h = ts.hour
    out = np.full(len(ts), 'Late', dtype=object)
    out[(h >= 0) & (h < 8)]   = 'Asia'
    out[(h >= 8) & (h < 13)]  = 'Europe'
    out[(h >= 13) & (h < 21)] = 'US'
    out[(h >= 21) & (h < 24)] = 'Late'
    return pd.Series(out, index=ts, name='session')

tod_rows = []
for (regime_name, strat_name), res in regime_results.items():
    sess = session_of(res.returns.index)
    for s in ['Asia', 'Europe', 'US', 'Late']:
        sub = res.returns[sess == s]
        if len(sub) < 20 or sub.std() == 0:
            sharpe = float('nan'); mean_r = float('nan')
        else:
            # Annualize on 'this session's bars/year' basis
            bars_per_year_session = {'Asia': 365*8, 'Europe': 365*5, 'US': 365*8, 'Late': 365*3}[s]
            mean_r = float(sub.mean())
            sharpe = float(sub.mean() / sub.std() * np.sqrt(bars_per_year_session))
        tod_rows.append({'regime': regime_name, 'strategy': strat_name,
                         'session': s, 'n_bars': int(len(sub)),
                         'mean_ret_per_bar_bps': mean_r * 1e4,
                         'session_Sharpe': sharpe})
tod_df = pd.DataFrame(tod_rows)

# Show the two surviving strategies only (the ones whose session split matters most)
for strat in ['funding_momentum', 'funding_contrarian']:
    print(f'=== {strat}: session_Sharpe by regime ===')
    pivot = tod_df[tod_df.strategy == strat].pivot(
        index='session', columns='regime', values='session_Sharpe'
    )[['bear_2022Q1','chop_2023H1','bull_2024H1']]
    pivot = pivot.reindex(['Asia','Europe','US','Late'])
    print(pivot)
    print()

In [ ]:
# Where does mean-reversion work best by session?
for strat in ['bollinger_mr_threshold', 'rsi_mr_threshold']:
    print(f'=== {strat}: session_Sharpe by regime ===')
    pivot = tod_df[tod_df.strategy == strat].pivot(
        index='session', columns='regime', values='session_Sharpe'
    )[['bear_2022Q1','chop_2023H1','bull_2024H1']]
    pivot = pivot.reindex(['Asia','Europe','US','Late'])
    print(pivot)
    print()

## 5. P&L control overlay on funding_momentum

Add stateful stop-loss / profit-take / time-out. Implemented as a post-process on the weight series. The overlay tracks entry price per symbol; exits force weight to 0 for a cooldown period.

In [ ]:
def apply_pnl_control(
    weights: pd.DataFrame,
    prices: pd.DataFrame,
    *,
    stop_loss_pct: float | None = None,
    profit_take_pct: float | None = None,
    max_hold_bars: int | None = None,
    cooldown_bars: int = 0,
) -> pd.DataFrame:
    """Stateful overlay: enter when weight sign changes; exit on stop/take/timeout;
    cooldown forces weight to 0 for N bars after an exit.
    
    For long positions: exit if (price/entry - 1) < -stop_loss_pct (stop) or > profit_take_pct (take).
    For short: exit if (entry/price - 1) < -stop_loss_pct or > profit_take_pct.
    """
    out = weights.copy()
    for sym in weights.columns:
        w_in = weights[sym].values
        p = prices[sym].values
        n = len(w_in)
        w_out = np.zeros(n)
        entry_price = None
        entry_sign = 0
        bars_held = 0
        cooldown = 0
        for t in range(n):
            if cooldown > 0:
                w_out[t] = 0.0
                cooldown -= 1
                continue
            desired = w_in[t]
            desired_sign = np.sign(desired)
            # If we have a position, check exits first using THIS bar's price
            if entry_sign != 0 and entry_price is not None:
                pnl = entry_sign * (p[t] / entry_price - 1)
                stop_hit = (stop_loss_pct is not None) and (pnl < -stop_loss_pct)
                take_hit = (profit_take_pct is not None) and (pnl > profit_take_pct)
                time_hit = (max_hold_bars is not None) and (bars_held >= max_hold_bars)
                if stop_hit or take_hit or time_hit:
                    w_out[t] = 0.0
                    entry_sign = 0; entry_price = None; bars_held = 0
                    cooldown = cooldown_bars
                    continue
            # Handle desired weight
            if desired_sign != entry_sign:
                # Sign change → exit (if any) and enter new
                if desired_sign != 0:
                    entry_price = p[t]; entry_sign = desired_sign; bars_held = 0
                    w_out[t] = desired
                else:
                    entry_price = None; entry_sign = 0; bars_held = 0
                    w_out[t] = 0.0
            else:
                w_out[t] = desired
                if entry_sign != 0:
                    bars_held += 1
        out[sym] = w_out
    return out

# Test on funding_momentum across regimes
pnl_variants = {
    'vanilla':          dict(),
    'sl_5pct':          dict(stop_loss_pct=0.05),
    'sl_10pct':         dict(stop_loss_pct=0.10),
    'pt_20pct':         dict(profit_take_pct=0.20),
    'sl5_pt20':         dict(stop_loss_pct=0.05, profit_take_pct=0.20),
    'sl5_pt20_to168h':  dict(stop_loss_pct=0.05, profit_take_pct=0.20, max_hold_bars=168),
    'timeout_7d':       dict(max_hold_bars=168),
    'timeout_30d':      dict(max_hold_bars=720),
}

pnl_rows = []
for regime_name, rd in regime_data.items():
    base_w = build_strategies(rd)['funding_momentum']().reindex(rd['common']).fillna(0.0)
    for vname, kwargs in pnl_variants.items():
        w = apply_pnl_control(base_w, rd['prices'], cooldown_bars=24, **kwargs)
        res = run_backtest(w, rd['prices'],
                           costs_bps=STRESS.fee_bps, slippage_bps=STRESS.slip_bps,
                           funding=rd['funding'], bars_per_year=BPY_1H)
        stats = summary(res.returns, periods=BPY_1H)
        pnl_rows.append({
            'regime': regime_name, 'variant': vname,
            'net_total': float((1+res.returns).prod()-1),
            'Sharpe': stats.get('Sharpe', float('nan')),
            'Calmar': stats.get('Calmar', float('nan')),
            'MaxDD':  stats.get('MaxDD', float('nan')),
            'avg_hold_h': holding_period_hours(w),
        })
pnl_df = pd.DataFrame(pnl_rows)
print('=== funding_momentum + P&L control: Sharpe by (variant, regime) ===')
print(pnl_df.pivot(index='variant', columns='regime', values='Sharpe')
          [['bear_2022Q1','chop_2023H1','bull_2024H1']]
          .reindex(list(pnl_variants.keys())))

In [ ]:
print('=== Net total return ===')
print(pnl_df.pivot(index='variant', columns='regime', values='net_total')
          [['bear_2022Q1','chop_2023H1','bull_2024H1']]
          .reindex(list(pnl_variants.keys())))
print()
print('=== avg holding hours ===')
print(pnl_df.pivot(index='variant', columns='regime', values='avg_hold_h')
          [['bear_2022Q1','chop_2023H1','bull_2024H1']]
          .reindex(list(pnl_variants.keys())))

## 6. ML signal ensemble (meta-model on strategy signals)

Use the SIGNAL series of each non-ML strategy as features (after lagging by 1 bar to ensure no look-ahead). Target = forward 1h direction. Train walk-forward on the combined regime timeline (3 regimes concatenated). Promote only if the ensemble beats `funding_momentum` in every regime.

In [ ]:
# Build per-symbol signal-feature matrix concatenated across all regimes
strat_names_for_ensemble = list(build_strategies(regime_data['bear_2022Q1']).keys())

def build_signal_features(rd):
    strategies = build_strategies(rd)
    feats = {}
    for sym in SYMBOLS:
        F = {}
        for sname, fn in strategies.items():
            w = fn()
            # The signal is the weight (already scaled to ±CAP), lagged 1 to ensure no look-ahead
            F[sname] = w[sym].shift(1)
        F['hour']   = pd.Series(rd['common'].hour, index=rd['common']).astype(float)
        F['dow']    = pd.Series(rd['common'].dayofweek, index=rd['common']).astype(float)
        feats[sym] = pd.DataFrame(F)
    return feats

all_features  = {sym: [] for sym in SYMBOLS}
all_labels    = {sym: [] for sym in SYMBOLS}
all_returns   = {sym: [] for sym in SYMBOLS}
regime_tag    = {sym: [] for sym in SYMBOLS}
for regime_name, rd in regime_data.items():
    feats = build_signal_features(rd)
    for sym in SYMBOLS:
        F = feats[sym]
        ret = rd['prices'][sym].pct_change()
        fwd = ret.shift(-1)
        y = (fwd > 0).astype(int)
        all_features[sym].append(F)
        all_labels[sym].append(y)
        all_returns[sym].append(fwd)
        regime_tag[sym].append(pd.Series(regime_name, index=F.index))

X = {sym: pd.concat(all_features[sym]).sort_index() for sym in SYMBOLS}
y = {sym: pd.concat(all_labels[sym]).sort_index() for sym in SYMBOLS}
fwd_ret_all = {sym: pd.concat(all_returns[sym]).sort_index() for sym in SYMBOLS}
regime_idx = {sym: pd.concat(regime_tag[sym]).sort_index() for sym in SYMBOLS}

for sym in SYMBOLS:
    print(f'{sym}: X={X[sym].shape}, regimes covered={regime_idx[sym].unique().tolist()}')

In [ ]:
wf = WalkForwardSplit(train_size='90D', val_size='30D', step='30D', mode='expanding')

ml_models = {
    'logistic': lambda: LogisticRegression(C=1.0, max_iter=500),
    'lightgbm': lambda: lgb.LGBMClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, verbose=-1),
}

def wf_predict(factory, X_full, y_full):
    preds = pd.Series(np.nan, index=X_full.index)
    for split in wf.split(X_full.index):
        X_tr = X_full.loc[split.train].dropna()
        y_tr = y_full.loc[X_tr.index]
        X_va = X_full.loc[split.val].dropna()
        y_va_idx = y_full.loc[X_va.index].dropna().index
        X_va = X_va.loc[y_va_idx]
        if len(X_tr) < 200 or len(X_va) < 50: continue
        sc = Standardizer(mode='per_column').fit(X_tr)
        X_tr_s = np.nan_to_num(sc.transform(X_tr).values)
        X_va_s = np.nan_to_num(sc.transform(X_va).values)
        m = factory()
        m.fit(X_tr_s, y_tr.values)
        preds.loc[X_va.index] = m.predict_proba(X_va_s)[:, 1]
    return preds

ensemble_preds = {name: {} for name in ml_models}
for sym in SYMBOLS:
    X_sym = X[sym].copy()
    y_sym = y[sym].copy()
    common_idx_ml = X_sym.dropna().index.intersection(y_sym.dropna().index)
    X_sym = X_sym.loc[common_idx_ml]
    y_sym = y_sym.loc[common_idx_ml]
    for name, factory in ml_models.items():
        ensemble_preds[name][sym] = wf_predict(factory, X_sym, y_sym)
    print(f'{sym}: ensemble predictions length = {len(ensemble_preds["logistic"][sym].dropna())}')

In [ ]:
# Convert predictions to weights, slice into regimes, backtest each
def preds_to_w(preds_dict, threshold=0.55):
    cols = {}
    for sym in SYMBOLS:
        p = preds_dict[sym]
        w = pd.Series(0.0, index=p.index)
        w[p > threshold] = CAP
        w[p < 1 - threshold] = -CAP
        cols[sym] = w
    return pd.DataFrame(cols)

ml_rows = []
for name in ml_models:
    w_full = preds_to_w(ensemble_preds[name])
    for regime_name, rd in regime_data.items():
        w_r = w_full.reindex(rd['common']).fillna(0.0)
        res = run_backtest(
            w_r, rd['prices'],
            costs_bps=STRESS.fee_bps, slippage_bps=STRESS.slip_bps,
            funding=rd['funding'], bars_per_year=BPY_1H,
        )
        stats = summary(res.returns, periods=BPY_1H)
        ml_rows.append({
            'model': name, 'regime': regime_name,
            'net_total': float((1+res.returns).prod()-1),
            'Sharpe': stats.get('Sharpe', float('nan')),
            'Calmar': stats.get('Calmar', float('nan')),
            'avg_hold_h': holding_period_hours(w_r),
        })
ml_df = pd.DataFrame(ml_rows)
print('=== ML ensemble: Sharpe by (model, regime) ===')
print(ml_df.pivot(index='model', columns='regime', values='Sharpe')
          [['bear_2022Q1','chop_2023H1','bull_2024H1']])
print()
print('=== ML ensemble: net total ===')
print(ml_df.pivot(index='model', columns='regime', values='net_total')
          [['bear_2022Q1','chop_2023H1','bull_2024H1']])

## 7. Promotion test: does ML beat `funding_momentum` in EVERY regime?

In [ ]:
fm_sharpe = rg_df[rg_df.strategy=='funding_momentum'].set_index('regime')['Sharpe']
print('funding_momentum Sharpe per regime:')
print(fm_sharpe)
print()
for model in ml_models:
    ml_s = ml_df[ml_df.model==model].set_index('regime')['Sharpe']
    beats = (ml_s > fm_sharpe).all() and (ml_s.notna().all())
    print(f'{model}: beats funding_momentum in every regime? {beats}')
    for r in fm_sharpe.index:
        print(f'   {r}: ml={ml_s.get(r, float("nan")):.3f}  vs  fm={fm_sharpe[r]:.3f}')

## 8. PM holdout audit

In [ ]:
audit = access_summary_for_report()
for k, v in audit.items():
    print(f'  {k}: {v}')
if audit['accessed']:
    raise RuntimeError('PM HOLDOUT WAS ACCESSED — contaminated.')
print('\nPM Holdout was not accessed.')